In [2]:
print('Ritu')

Ritu


In [1]:
from langchain_ai21.chat_models import ChatAI21
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import SystemMessage,BaseMessage,HumanMessage,AIMessage
from langgraph.types import interrupt,Command 
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode,tools_condition
from typing import TypedDict,Annotated, List, Dict, Literal
from langgraph.checkpoint.postgres import PostgresSaver
from psycopg_pool import ConnectionPool

from dotenv import load_dotenv
import json
import sqlite3
import os


load_dotenv()

model = ChatAI21(model = 'jamba-mini-1.7-2025-07')
searchTool = TavilySearchResults(max_results=3)
tools = [searchTool]
model_with_tools = model.bind_tools(tools)


C:\Users\kaushal\AppData\Local\Temp\ipykernel_3208\16932131.py:22: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  searchTool = TavilySearchResults(max_results=3)


In [3]:
# model = ChatAI21(
#     model="jamba-mini-1.7-2025-07",
#     # api_key="cc97c84e-8137-476c-8324-ce151112c72d",
#     # base_url="https://api.ai21.com/studio/v1"
# )
model.invoke('Hi')

AIMessage(content='Hi! How can I assist you today?', additional_kwargs={}, response_metadata={}, id='lc_run--019bf8bd-c162-7953-9ff0-7f90d1cf8127-0', tool_calls=[], invalid_tool_calls=[])

# Simple Flow

In [ ]:
OUTLINE_SYSTEM_PORMPT = SystemMessage(
    content="""
You are an expert presentation designer and content strategist.

Your task: Create a STRUCTURED OUTLINE for a PowerPoint presentation.

Output format (JSON):
{
    "title":"Main presentation title",
    "totle_slides": number,
    "slides":[
        { 
            "slide_number":1,
            "slide_title": "Title of the slide"
            "key_points": ["Point 1","Point 2","Point 3"]
            "content_type": "introduction/explanation/comparison/conclusion"
        }
    ]
}

Rules:
- Create a logical flow from introduction to conclusion
- Each slide should have 3-5 key points
- Be specific about what each slide will cover
- Ensure comprehensive coverage of the topic
- Use tools to research if needed for accuracy
- Return ONLY valid JSON, no additional text
"""
)


def except_outline_prompt(outline_text):
    prompt = HumanMessage(
            content=f"""The following text should be valid JSON but has formatting issues:

{outline_text}

Please convert this into valid JSON with the following structure:
{{
    "title":"Main presentation title",
    "totle_slides": number,
    "slides":[
        {{ 
            "slide_number":1,
            "slide_title": "Title of the slide"
            "key_points": ["Point 1","Point 2","Point 3"]
            "content_type": "introduction/explanation/comparison/conclusion"
        }}
    ]
}}

Return ONLY the valid JSON, no explanations or markdown formatting."""
        )
    return prompt 

In [ ]:
DETAIL_SYSTEM_PROMPT = SystemMessage(
    content="""
You are an expert content writer for presentations.

Your task: Generate DETAILED, ENGAGING content for a specific slide.

For each key point:
- Provide 2-3 sentences of explanation
- Include relevant examples, statistics, or facts
- Make it clear, concise, and presentation-ready
- Use simple language that's easy to understand

Output format:
{
    "slide_number": number,
    "slide_title": "Title",
    "detailed_content": [
        {
            "key_point":
        }
    ]
}

"""
)

def except_detail_prompt(detail_text):
    prompt = HumanMessage(
            content=f"""The following text should be valid JSON but has formatting issues:

{detail_text}

Please convert this into valid JSON with the following structure:
{{
    "slide_number": number,
    "slide_title": "Title",
    "detailed_content": [
        {{
            "key_point":
        }}
    ]
}}

Return ONLY the valid JSON, no explanations or markdown formatting."""
        )
    return prompt

In [ ]:
def is_valid_brackets(s: str) -> bool:
     if s == '':
          return
     stack = []
     bracket_map = {
         '(' : ')',
         '{' : '}',
         '[' : ']',
     }

     if s[0] != '{':
         s = '{' +s

     for char in s:
        if char in bracket_map:
            stack.append(char)

        elif char in bracket_map.values():
                stack.pop()

     if s[-1] not in ['"',']','}']:
         s = s+'"'
     for char in stack:
         s += bracket_map[char]
     return s

In [ ]:
def json_to_python(fixed_text):
    if "```json" in fixed_text:
        fixed_text = fixed_text.replace("```json", "").replace("```", "").strip()
    elif "```" in fixed_text:
        fixed_text = fixed_text.replace("```", "").strip()
    return fixed_text


In [ ]:
class PptState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    outline: Dict
    detailed_slides: List[Dict]
    current_slide_index: int
    topic: str
    num_slides: int

In [ ]:
def chat_node(state: PptState):
    """Main chat node that processess messages"""
    messages = state["messages"]
    result = model_with_tools.invoke(messages)
    return {'messages':result}

In [ ]:
def generate_outline_node(state: PptState):
    """Step 1: Generate presentation outline"""
    topic = state["topic"]
    num_slides = state['num_slides']

    prompt = HumanMessage(
        content=f"Create a {num_slides}-slide presentation outline on: {topic}"
    )

    messages = [OUTLINE_SYSTEM_PORMPT,prompt]
    result = model_with_tools.invoke(messages)
    # print('generate_outline_node result',type(result.content))
    try:
        outline = json_to_python(result.content)
        outline = json.loads(outline)
    except json.JSONDecodeError:
        
        outline = json.loads(is_valid_brackets(outline))
        # fix_prompt = except_outline_prompt(result.content)
        # try:
        #     fix_result = model.invoke([fix_prompt])
        #     fixed_text = fix_result.content.strip()
        #     outline = json_to_python(fixed_text)
        #     outline = json.loads(outline)
        # except json.JSONDecodeError:
        #     outline = {
        #         'title': topic,
        #         'total_slide': num_slides,
        #         'slides': []
        #     }
    print('outline',outline['slides'])

    return {
        'messages':[result],
        'outline':outline,
        'current_slide_index':0
            }

In [ ]:
def generate_slide_detail_node(state: PptState):
    """Step 2: Generate detailed content for current Slide"""
    outline = state['outline']
    current_index = state['current_slide_index']
    if current_index > state['num_slides']:
        return
    
    # print('outline',type(outline),outline)
    print('current_index',current_index)
    current_slide = outline['slides'][int(current_index)]
    # print('current_slide',current_slide)
    # print('slide_title',current_slide['slide_title'])
    # print('key_points',current_slide['key_points'])
    # print('content_type',current_slide['content_type'])

    prompt = HumanMessage(
        content=f"""Generate detailede content for this slide:
Slide Number: {current_slide['slide_title']}
Key Points: {', '.join(current_slide['key_points'])}
Content Type: {current_slide['content_type']}

Provide comprehensive, presentation-ready content."""
    )

    messages = [DETAIL_SYSTEM_PROMPT,prompt]
    result = model_with_tools.invoke(messages)
    # print('result.content',result.content)

    try:
        # print('inside first try')
        detailed_slide = json_to_python(result.content)
        detailed_slide = json.loads(detailed_slide)
        
    except json.JSONDecodeError as e:
        print('before is_valid_brackets',detailed_slide)
        print('after is_valid_brackets',is_valid_brackets(detailed_slide))
        try:
            detailed_slide = json.loads(is_valid_brackets(detailed_slide))
        except:
            fix_prompt = except_detail_prompt(detailed_slide)
            try:
                fix_result = model.invoke([fix_prompt])
                fixed_text = fix_result.content.strip()
                detailed_slide = json_to_python(fixed_text)
                detailed_slide = json.loads(is_valid_brackets(detailed_slide))
            except json.JSONDecodeError as e:
                print('last erroe',e)
                print('detailed_slide',detailed_slide)
                detailed_slide = {
                    'slide_number': current_slide['slide_number'],
                    'slide_title': current_slide['slide_title'],
                    'detailed_content':[],
                    'row_response': detailed_slide
                }
    
    detailed_slides = state.get('detailed_slides',[])
    detailed_slides.append(detailed_slide)
    return {
        'messages':[result],
        'detailed_slides':detailed_slides,
        'current_slide_index': current_index +1
    }

In [ ]:
def should_coutine_slides(state: PptState):
    """Check if we need to generate more slides"""
    current_index = state.get('current_slide_index',0)
    total_slides = len(state.get('outline',{}).get('slides',[]))

    if current_index < total_slides:
        return "continue"
    else:
        return "end"

In [ ]:
tool_node = ToolNode(tools)

workflow = StateGraph(PptState)
workflow.add_node('generate_outline',generate_outline_node)
workflow.add_node("generate_slide_detail",generate_slide_detail_node)
workflow.add_node('chat_node',chat_node)
workflow.add_node('tools',tool_node)

workflow.add_edge(START,'generate_outline')
workflow.add_edge("generate_outline","generate_slide_detail")
workflow.add_conditional_edges(
    "generate_slide_detail",should_coutine_slides,
    {
        "continue":"generate_slide_detail",
        "end":END
    }

)

conn = sqlite3.connect("graph.db", check_same_thread=False)
checkpointer = SqliteSaver(conn)

ppt_generator = workflow.compile(checkpointer = checkpointer)

In [ ]:
initial_state = {
        'messages': [],
        'outline': {},
        'detailed_slides': [],
        'current_slide_index': 0,
        'topic': 'what is recursion in python',
        'num_slides': 1
    }
config = {"configurable":{"thread_id":"user-123"}}
result = ppt_generator.invoke(initial_state,config=config)

In [ ]:
result

# HITL

In [4]:
OUTLINE_SYSTEM_PORMPT = SystemMessage(
    content="""
You are an expert presentation designer and content strategist.

Your task: Create a STRUCTURED OUTLINE for a PowerPoint presentation.

Output format (JSON):
{
    "title":"Main presentation title",
    "totle_slides": number,
    "slides":[
        { 
            "slide_number":1,
            "slide_title": "Title of the slide"
            "key_points": ["Point 1","Point 2","Point 3"]
            "content_type": "introduction/explanation/comparison/conclusion"
        }
    ]
}

Rules:
- Create a logical flow from introduction to conclusion
- Each slide should have 3-5 key points
- Be specific about what each slide will cover
- Ensure comprehensive coverage of the topic
- Use tools to research if needed for accuracy
- Return ONLY valid JSON, no additional text
"""
)


def except_outline_prompt(outline_text):
    prompt = HumanMessage(
            content=f"""The following text should be valid JSON but has formatting issues:

{outline_text}

Please convert this into valid JSON with the following structure:
{{
    "title":"Main presentation title",
    "totle_slides": number,
    "slides":[
        {{ 
            "slide_number":1,
            "slide_title": "Title of the slide"
            "key_points": ["Point 1","Point 2","Point 3"]
            "content_type": "introduction/explanation/comparison/conclusion"
        }}
    ]
}}

Return ONLY the valid JSON, no explanations or markdown formatting."""
        )
    return prompt 


DETAIL_SYSTEM_PROMPT = SystemMessage(
    content="""
You are an expert content writer for presentations.

Your task: Generate DETAILED, ENGAGING content for a specific slide.

For each key point:
- Provide 2-3 sentences of explanation
- Include relevant examples, statistics, or facts
- Make it clear, concise, and presentation-ready
- Use simple language that's easy to understand

Output format:
{
    "slide_number": number,
    "slide_title": "Title",
    "detailed_content": [
        {
            "key_point":
        }
    ]
}

"""
)

def except_detail_prompt(detail_text):
    prompt = HumanMessage(
            content=f"""The following text should be valid JSON but has formatting issues:

{detail_text}

Please convert this into valid JSON with the following structure:
{{
    "slide_number": number,
    "slide_title": "Title",
    "detailed_content": [
        {{
            "key_point":
        }}
    ]
}}

Return ONLY the valid JSON, no explanations or markdown formatting."""
        )
    return prompt

def is_valid_brackets(s: str):
     if s == '':
          return
     stack = []
     bracket_map = {
         '(' : ')',
         '{' : '}',
         '[' : ']',
     }

     if s[0] != '{':
         s = '{' +s

     for char in s:
        if char in bracket_map:
            stack.append(char)

        elif char in bracket_map.values():
                stack.pop()

     if s[-1] not in ['"',']','}']:
         s = s+'"'
     for char in stack[::-1]:
         s += bracket_map[char]
     return s

def json_to_python(fixed_text):
    if "```json" in fixed_text:
        fixed_text = fixed_text.replace("```json", "").replace("```", "").strip()
    elif "```" in fixed_text:
        fixed_text = fixed_text.replace("```", "").strip()
    return fixed_text

class PptState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    outline: Dict
    detailed_slides: List[Dict]
    current_slide_index: int
    topic: str
    feedback: str
    action: str
    num_slides: int

def chat_node(state: PptState):
    """Main chat node that processess messages"""
    messages = state["messages"]
    result = model_with_tools.invoke(messages)
    return {'messages':result}


def generate_outline_node(state: PptState):
    """Step 1: Generate presentation outline"""
    topic = state["topic"]
    num_slides = state['num_slides']

    prompt = HumanMessage(
        content=f"Create a {num_slides}-slide presentation outline on: {topic}"
    )

    messages = [OUTLINE_SYSTEM_PORMPT,prompt]
    result = model_with_tools.invoke(messages)
    # print('generate_outline_node result',type(result.content))
    try:
        outline = json_to_python(result.content)
        outline = json.loads(outline)
    except json.JSONDecodeError:
        
        outline = json.loads(is_valid_brackets(outline))

    return {
        'messages':[result],
        'outline':outline,
        'current_slide_index':0
            }

def generate_slide_detail_node(state: PptState):
    """Step 2: Generate detailed content for current Slide"""
    outline = state['outline']
    current_index = state['current_slide_index']
    if current_index > state['num_slides']:
        return
    
    print('current_index',current_index)
    current_slide = outline['slides'][int(current_index)]

    prompt = HumanMessage(
        content=f"""Generate detailede content for this slide:
Slide Number: {current_slide['slide_title']}
Key Points: {', '.join(current_slide['key_points'])}
Content Type: {current_slide['content_type']}

Provide comprehensive, presentation-ready content."""
    )

    messages = [DETAIL_SYSTEM_PROMPT,prompt]
    result = model_with_tools.invoke(messages)
    # print('result.content',result.content)

    try:
        # print('inside first try')
        detailed_slide = json_to_python(result.content)
        detailed_slide = json.loads(detailed_slide)
        
    except json.JSONDecodeError as e:
        print('before is_valid_brackets',detailed_slide)
        print('after is_valid_brackets',is_valid_brackets(detailed_slide))
        try:
            detailed_slide = json.loads(is_valid_brackets(detailed_slide))
        except:
            fix_prompt = except_detail_prompt(detailed_slide)
            try:
                fix_result = model.invoke([fix_prompt])
                fixed_text = fix_result.content.strip()
                detailed_slide = json_to_python(fixed_text)
                detailed_slide = json.loads(is_valid_brackets(detailed_slide))
            except json.JSONDecodeError as e:
                print('last erroe',e)
                print('detailed_slide',detailed_slide)
                detailed_slide = {
                    'slide_number': current_slide['slide_number'],
                    'slide_title': current_slide['slide_title'],
                    'detailed_content':[],
                    'row_response': detailed_slide
                }
    
    detailed_slides = state.get('detailed_slides',[])
    detailed_slides.append(detailed_slide)
    return {
        'messages':[result],
        'detailed_slides':detailed_slides,
        'current_slide_index': current_index +1
    }

def should_coutine_slides(state: PptState):
    """Check if we need to generate more slides"""
    current_index = state.get('current_slide_index',0)
    total_slides = len(state.get('outline',{}).get('slides',[]))

    if current_index < total_slides:
        return "continue"
    else:
        return "end"

def human_decision(state: PptState):
    pass
    
tool_node = ToolNode(tools)

workflow = StateGraph(PptState)
workflow.add_node('generate_outline',generate_outline_node)
workflow.add_node("generate_slide_detail",generate_slide_detail_node)
workflow.add_node('chat_node',chat_node)
workflow.add_node('tools',tool_node)

workflow.add_edge(START,'generate_outline')
workflow.add_edge("generate_outline","generate_slide_detail")
workflow.add_conditional_edges(
    "generate_slide_detail",should_coutine_slides,
    {
        "continue":"generate_slide_detail",
        "end":END
    }

)

conn = sqlite3.connect("graph.db", check_same_thread=False)
checkpointer = SqliteSaver(conn)

ppt_generator = workflow.compile(checkpointer = checkpointer)


# initial_state = {
#         'messages': [],
#         'outline': {},
#         'detailed_slides': [],
#         'current_slide_index': 0,
#         'topic': 'What is photosynthesis in plants',
#         'num_slides': 1
#     }
# config = {"configurable":{"thread_id":"user-123"}}
# result = ppt_generator.invoke(initial_state,config=config)

# HITL IN OUTLINE

In [12]:
OUTLINE_SYSTEM_PORMPT = SystemMessage(
    content="""
You are an expert presentation designer and content strategist.

Your task: Create a STRUCTURED OUTLINE for a PowerPoint presentation.

Output format (JSON):
{
    "title":"Main presentation title",
    "totle_slides": number,
    "slides":[
        { 
            "slide_number":1,
            "slide_title": "Title of the slide"
            "key_points": ["Point 1","Point 2","Point 3"]
            "content_type": "introduction/explanation/comparison/conclusion"
        }
    ]
}

Rules:
- Create a logical flow from introduction to conclusion
- Each slide should have 3-5 key points
- Be specific about what each slide will cover
- Ensure comprehensive coverage of the topic
- Use tools to research if needed for accuracy
- Return ONLY valid JSON, no additional text
"""
)


class PptState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    outline: Dict
    detailed_slides: List[Dict]
    current_slide_index: int
    feedback: str
    action: Literal['continue_slide',"continue_next",'update_outline','update_slide']
    


def generate_outline_node(state: PptState):
    """Step 1: Generate presentation outline"""

    print('inside generate_outline_node')
    messages = state["messages"] + [OUTLINE_SYSTEM_PORMPT]
    result = model_with_tools.invoke(messages)
    print('result',result)
    
    output = {
        'messages':[result],
        'current_slide_index':0
            } 

    if result.content:
        try:
            outline = json_to_python(result.content)
            outline = json.loads(outline)
        except json.JSONDecodeError as e:
            try:
                outline = json.loads(is_valid_brackets(outline))
                print('outline 3',outline)
            except json.JSONDecodeError as e:
                print('generate_outline_node',e)
        output['outline'] = outline
        
    print("state before ",state)
    return output

def generate_slide_detail_node(state: PptState):
    return {
        'messages':[AIMessage('this is generate_slide_detail_node')],
        'detailed_slides':{'ritu':'Ritu this is generate_slide_detail_node'},
        'current_slide_index': 1
    }

def human_decision(state: PptState):
    decision = interrupt({})
    print('inside human_decision')
    print('decision',decision)
    print('state',state)

    if decision['action'] == "update_outline":
        return {
            'action': "update_outline",
            "messages":[decision['feedback']]
            }
        # print('inside first 1')

        # if 'feedback' in  decision:
        #     print('inside first 2')
        #     if 'add_slide' in decision:
        #         print('inside first 3')
        #         # need to improve
        #         # return {
        #         #     'action': "update_outline",
        #         #     "feedback":decision['feedback'],
        #         #     "num_slides":int(decision['add_slide'])+state['num_slides'],
        #         #     "add_slide":int(decision['add_slide'])
        #         #     }
        #     elif 'remove_slide' in decision:
        #         print('inside first 4')
        #         # # need to improve
        #         # remove_slides(decision['remove_slide'],state['outline'])
        #         # # state['outline']['totle_slides'] -= len(decision['remove_slide'].split(','))
        #         # return {
        #         #     'action': "update_outline",
        #         #     "remove_slide":decision['remove_slide'],
        #         #     "feedback":decision['feedback'],
        #         #     "num_slides":state['num_slides']- len(decision['remove_slide'].split(','))
        #         #     }
        #     else:
        
        # elif 'add_slide' in decision:
        #     print('inside first 5')
        #     return {
        #         'action': "update_outline",
        #         "feedback":'add_slide',
        #         "num_slides":int(decision['add_slide'])+state['num_slides'],
        #         "add_slide":int(decision['add_slide'])
        #         }
        # elif 'remove_slide' in decision:
        #     print('inside first 6')
        #     # remove_slides(decision['remove_slide'],state['outline']['slides'])
        #     remove_slides(decision['remove_slide'],state['outline'])
        #     # state['outline']['totle_slides'] -= len(decision['remove_slide'].split(','))
        #     print('state',state)
        #     return {
        #         'action': "update_outline",
        #         "remove_slide":decision['remove_slide'],
        #         "feedback":'remove_slide',
        #         "num_slides":state['num_slides']- len(decision['remove_slide'].split(','))
        #         }
    elif decision['action'] == 'continue_slide':
        return {'action':'continue_slide'}

    
def route_after_human(state: PptState):
    action = state['action']
    if action == 'update_outline':
        return "update_outline"

    elif action in ('continue_slide', 'continue_next', 'update_slide'):
        return "continue_slide"

DB_URL = os.getenv("ppt_url")

tool_node = ToolNode(tools)

workflow = StateGraph(PptState)
workflow.add_node('generate_outline',generate_outline_node)
workflow.add_node('human_decision',human_decision)
workflow.add_node('generate_slide_detail',generate_slide_detail_node)
# workflow.add_node('chat_node',chat_node)
workflow.add_node('tools',tool_node)

workflow.add_edge(START,'generate_outline')
# workflow.add_conditional_edges('generate_outline',tools_condition)
workflow.add_conditional_edges(
    "generate_outline",
    tools_condition,
    {
        "tools": "tools",
        "__end__": "human_decision",
    },
)

workflow.add_edge("tools", "generate_outline")
workflow.add_conditional_edges('human_decision',
    route_after_human,{
    "update_outline":"generate_outline",
    "continue_slide":"generate_slide_detail",
})
workflow.add_edge('generate_slide_detail',END)


from langgraph.checkpoint.postgres import PostgresSaver
from psycopg_pool import ConnectionPool

connection_kwargs = {
        "autocommit": True,
        "prepare_threshold": 0,
    }

pool = ConnectionPool(
       conninfo=DB_URL,
        max_size=20,
        kwargs=connection_kwargs,
)

checkpointer = PostgresSaver(pool)
checkpointer.setup()
graph = workflow.compile(checkpointer=checkpointer)


In [18]:
# # content=f"Create a {num_slides}-slide presentation outline on: {topic}
# num_slides = 3
# topic = 'Rest API'
# result = graph.invoke({"messages":HumanMessage(content=f"Create a {num_slides}-slide presentation outline on: {topic}"),
#                       "num_slides":num_slides},
#                       config = {"configurable":{"thread_id":118}}
# )

In [19]:
# input_data = Command(
#     resume={
#         "action":'update_outline',
#         "feedback":'can you add one more slide to the outline'
        
#     }
# )
# result1 = graph.invoke(input_data,
#     config = {"configurable":{"thread_id":118}})

# HITL IN SLIDE

In [ ]:
class PptState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    outline: Dict
    detailed_slides: List[Dict]
    current_slide_index: int
    feedback: str
    action: Literal[
        "continue_slide",
        # "continue_next",
        "update_outline",
        "update_slide",
        ''
    ]
    tool_caller: Literal[
        "generate_outline",
        "generate_slide_detail"
    ]


OUTLINE_SYSTEM_PORMPT = SystemMessage(
    content="""
You are an expert presentation designer and content strategist.

Your task: Create a STRUCTURED OUTLINE for a PowerPoint presentation.

Output format (JSON):
{
    "title":"Main presentation title",
    "totle_slides": number,
    "slides":[
        { 
            "slide_number":1,
            "slide_title": "Title of the slide"
            "key_points": ["Point 1","Point 2","Point 3"]
            "content_type": "introduction/explanation/comparison/conclusion"
        }
    ]
}

Rules:
- Create a logical flow from introduction to conclusion
- Each slide should have 3-5 key points
- Be specific about what each slide will cover
- Ensure comprehensive coverage of the topic
- Use tools to research if needed for accuracy
- Return ONLY valid JSON, no additional text
"""
)


DETAIL_SYSTEM_PROMPT = SystemMessage(
    content="""
You are an expert content writer for presentations.

Your task: Generate DETAILED, ENGAGING content for a specific slide.

For each key point:
- Provide 2-3 sentences of explanation
- Include relevant examples, statistics, or facts
- Make it clear, concise, and presentation-ready
- Use simple language that's easy to understand

Output format:
{
    "slide_number": number,
    "slide_title": "Title",
    "detailed_content": [
        {
            "key_point":
        }
    ]
}

"""
)

def is_valid_brackets(s: str):
     if s == '':
          return
     stack = []
     bracket_map = {
         '(' : ')',
         '{' : '}',
         '[' : ']',
     }

     if s[0] != '{':
         s = '{' +s

     for char in s:
        if char in bracket_map:
            stack.append(char)

        elif char in bracket_map.values():
                stack.pop()

     if s[-1] not in ['"',']','}']:
         s = s+'"'
     for char in stack[::-1]:
         s += bracket_map[char]
     return s

def json_to_python(fixed_text):
    if "```json" in fixed_text:
        fixed_text = fixed_text.replace("```json", "").replace("```", "").strip()
    elif "```" in fixed_text:
        fixed_text = fixed_text.replace("```", "").strip()
    return fixed_text


def generate_outline_node(state: PptState):
    """Step 1: Generate presentation outline"""

    print('inside generate_outline_node')
    messages = state["messages"] + [OUTLINE_SYSTEM_PORMPT]
    result = model_with_tools.invoke(messages)
    print('result',result)
    
    output = {
        'messages':[result],
        'current_slide_index':0,
        "tool_caller": "generate_outline",
            } 

    if result.content:
        try:
            outline = json_to_python(result.content)
            outline = json.loads(outline)
        except json.JSONDecodeError as e:
            try:
                outline = json.loads(is_valid_brackets(outline))
                print('outline 3',outline)
            except json.JSONDecodeError as e:
                print('generate_outline_node',e)
        output['outline'] = outline
        
    print("state before ",state)
    return output


def generate_slide_detail_node(state: PptState):
    """Step 2: Generate detailed content for current Slide"""

    outline = state['outline']
    current_index = state['current_slide_index']
    detailed_slides = state.get('detailed_slides',[])
    total_slides = len(state.get('outline',{}).get('slides',[]))
    # if current_index > state['num_slides']:
    #     return
    if current_index > total_slides:
        # return
        return {"action": "end"}
    
    output = {
        "tool_caller": "generate_slide_detail",
    }

    if state['action'] == "update_slide":
        feedback = state['action']
        last_slide = detailed_slides.pop()
        last_outline = outline['slides'][current_index-1]
        output['feedback'] = ''
        output['action'] = ''

        prompt = HumanMessage(
            content=f"""
You are updating a single slide in a PowerPoint presentation.
Presentation Title:
{state['outline']['title']}

Outline of the slide:
{last_outline}

Current Slide Content:
{last_slide}

User Feedback:
{feedback}
"""
        )
    else:
        print('current_index',current_index)
        current_slide = outline['slides'][int(current_index)]
        output['current_slide_index'] = current_index +1
        print('first output',output)

        prompt = HumanMessage(
            content=f"""Generate detailede content for this slide:
Slide Number: {current_slide['slide_title']}
Key Points: {', '.join(current_slide['key_points'])}
Content Type: {current_slide['content_type']}

Provide comprehensive, presentation-ready content."""
    )

    messages = [DETAIL_SYSTEM_PROMPT,prompt]
    result = model_with_tools.invoke(messages)
    # print('result.content',result.content)

    try:
        # print('inside first try')
        detailed_slide = json_to_python(result.content)
        detailed_slide = json.loads(detailed_slide)
        
    except json.JSONDecodeError as e:
        # print('before is_valid_brackets',detailed_slide)
        # print('after is_valid_brackets',is_valid_brackets(detailed_slide))
        try:
            detailed_slide = json.loads(is_valid_brackets(detailed_slide))
        except json.JSONDecodeError as e:
            print('generate_slide_detail_node inside',e)
            detailed_slide = is_valid_brackets(detailed_slide)
            
    detailed_slides.append(detailed_slide)
    output['messages'] = [result]
    output['detailed_slides'] = detailed_slides
    print('first output',output)

    return output


def route_after_tools(state: PptState):
    return state["tool_caller"]


def human_decision(state: PptState):
    decision = interrupt({})
    print('inside human_decision')
    print('decision',decision)
    print('state',state)

    if decision['action'] == "update_outline":
        return {
            'action': "update_outline",
            "messages":[decision['feedback']]
            }
        
    elif decision['action'] == 'continue_slide':
        return {'action':'continue_slide'}
    
    elif decision['action'] == 'update_slide':
        return {'action':'update_slide'}
    

    
def route_after_human(state: PptState):
    action = state['action']
    if action == 'update_outline':
        return "generate_outline"

    elif action in ('continue_slide', 'update_slide'):
        return "generate_slide_detail"
    return END

DB_URL = os.getenv("ppt_url")

tool_node = ToolNode(tools)



workflow = StateGraph(PptState)

# --------------------
# Nodes
# --------------------
workflow.add_node("generate_outline", generate_outline_node)
workflow.add_node("generate_slide_detail", generate_slide_detail_node)
workflow.add_node("human_decision", human_decision)
workflow.add_node("tools", ToolNode(tools))

# --------------------
# Start
# --------------------
workflow.add_edge(START, "generate_outline")

# --------------------
# Outline → tools OR human
# --------------------
workflow.add_conditional_edges(
    "generate_outline",
    tools_condition,
    {
        "tools": "tools",
        "__end__": "human_decision",
    },
)

# --------------------
# Slide → tools OR human
# --------------------
workflow.add_conditional_edges(
    "generate_slide_detail",
    tools_condition,
    {
        "tools": "tools",
        "__end__": "human_decision",
    },
)

# --------------------
# Tools → SAME caller
# --------------------
workflow.add_conditional_edges(
    "tools",
    route_after_tools,
    {
        "generate_outline": "generate_outline",
        "generate_slide_detail": "generate_slide_detail",
    },
)

# --------------------
# Human decides next
# --------------------
workflow.add_conditional_edges(
    "human_decision",
    route_after_human,
    {
        "generate_outline": "generate_outline",
        "generate_slide_detail": "generate_slide_detail",
        END: END,
    },
)




connection_kwargs = {
        "autocommit": True,
        "prepare_threshold": 0,
    }

pool = ConnectionPool(
       conninfo=DB_URL,
        max_size=20,
        kwargs=connection_kwargs,
)

checkpointer = PostgresSaver(pool)
checkpointer.setup()
graph = workflow.compile(checkpointer=checkpointer)


In [7]:
num_slides = 3
topic = 'Impect of ai in helth care'
initial_state: PptState = {
    "messages": [HumanMessage(content=f"Create a {num_slides}-slide presentation outline on: {topic}")],
    "outline": {},
    "detailed_slides": [],
    "current_slide_index": 0,
    "feedback": "",
    "action": "",
    "tool_caller": "generate_outline",
}

In [8]:
# content=f"Create a {num_slides}-slide presentation outline on: {topic}
# num_slides = 3
# topic = 'Rest API'
result = graph.invoke(initial_state,
                      config = {"configurable":{"thread_id":121}}
)

inside generate_outline_node
result content='```json\n{\n    "title": "The Impact of AI in Healthcare",\n    "totle_slides": 3,\n    "slides": [\n        {\n            "slide_number": 1,\n            "slide_title": "Introduction to AI in Healthcare",\n            "key_points": [\n                "Overview of AI in Healthcare",\n                "Key Applications of AI in Healthcare",\n                "Benefits of AI in Healthcare"\n            ],\n            "content_type": "introduction"\n        },\n        {\n            "slide_number": 2,\n            "slide_title": "Challenges and Limitations of AI in Healthcare",\n            "key_points": [\n                "Data Privacy and Security Concerns",\n                "Ethical and Legal Issues",\n                "High Costs and Implementation Barriers"\n            ],\n            "content_type": "comparison"\n        },\n        {\n            "slide_number": 3,\n            "slide_title": "Future of AI in Healthcare",\n            "

In [9]:
result

{'messages': [HumanMessage(content='Create a 3-slide presentation outline on: Impect of ai in helth care', additional_kwargs={}, response_metadata={}, id='594037ec-23fa-4a31-9174-63652845471b'),
  AIMessage(content='```json\n{\n    "title": "The Impact of AI in Healthcare",\n    "totle_slides": 3,\n    "slides": [\n        {\n            "slide_number": 1,\n            "slide_title": "Introduction to AI in Healthcare",\n            "key_points": [\n                "Overview of AI in Healthcare",\n                "Key Applications of AI in Healthcare",\n                "Benefits of AI in Healthcare"\n            ],\n            "content_type": "introduction"\n        },\n        {\n            "slide_number": 2,\n            "slide_title": "Challenges and Limitations of AI in Healthcare",\n            "key_points": [\n                "Data Privacy and Security Concerns",\n                "Ethical and Legal Issues",\n                "High Costs and Implementation Barriers"\n            ]

In [21]:
# ["continue_slide","update_outline","update_slide",'']
input_data = Command(
    resume={
        "action":'continue_slide',
        
    }
)
result1 = graph.invoke(input_data,
    config = {"configurable":{"thread_id":121}})

inside human_decision
decision {'action': 'continue_slide'}
state {'messages': [HumanMessage(content='Create a 3-slide presentation outline on: Impect of ai in helth care', additional_kwargs={}, response_metadata={}, id='594037ec-23fa-4a31-9174-63652845471b'), AIMessage(content='```json\n{\n    "title": "The Impact of AI in Healthcare",\n    "totle_slides": 3,\n    "slides": [\n        {\n            "slide_number": 1,\n            "slide_title": "Introduction to AI in Healthcare",\n            "key_points": [\n                "Overview of AI in Healthcare",\n                "Key Applications of AI in Healthcare",\n                "Benefits of AI in Healthcare"\n            ],\n            "content_type": "introduction"\n        },\n        {\n            "slide_number": 2,\n            "slide_title": "Challenges and Limitations of AI in Healthcare",\n            "key_points": [\n                "Data Privacy and Security Concerns",\n                "Ethical and Legal Issues",\n       

In [22]:
result1

{'messages': [HumanMessage(content='Create a 3-slide presentation outline on: Impect of ai in helth care', additional_kwargs={}, response_metadata={}, id='594037ec-23fa-4a31-9174-63652845471b'),
  AIMessage(content='```json\n{\n    "title": "The Impact of AI in Healthcare",\n    "totle_slides": 3,\n    "slides": [\n        {\n            "slide_number": 1,\n            "slide_title": "Introduction to AI in Healthcare",\n            "key_points": [\n                "Overview of AI in Healthcare",\n                "Key Applications of AI in Healthcare",\n                "Benefits of AI in Healthcare"\n            ],\n            "content_type": "introduction"\n        },\n        {\n            "slide_number": 2,\n            "slide_title": "Challenges and Limitations of AI in Healthcare",\n            "key_points": [\n                "Data Privacy and Security Concerns",\n                "Ethical and Legal Issues",\n                "High Costs and Implementation Barriers"\n            ]

In [23]:
result1['detailed_slides']

['{ slide_number: 1\nslide_title: Introduction to AI in Healthcare\ndetailed_content:\n  - key_point:\n      overview_of_ai_in_healthcare:\n        - text: AI in healthcare refers to the use of advanced algorithms and machine learning models to analyze data, improve decision-making, and enhance patient care.\n        - text: It encompasses a wide range of applications, from diagnostics and treatment planning to administrative tasks and patient monitoring.\n        - text: Example: AI-powered tools like IBM Watson are used for cancer diagnosis and treatment recommendations.\n  - key_point:\n      key_applications_of_ai_in_healthcare:\n        - text: AI is applied in various healthcare sectors, including diagnostics, drug discovery, personalized medicine, and robotic surgery.\n       - text: For instance, AI models can analyze medical images to detect anomalies such as tumors or fractures with high accuracy.\n        - text: Example: AI systems like DeepMind\'s AlphaFold have revolution

In [24]:
print(result1['detailed_slides'][0])

{ slide_number: 1
slide_title: Introduction to AI in Healthcare
detailed_content:
  - key_point:
      overview_of_ai_in_healthcare:
        - text: AI in healthcare refers to the use of advanced algorithms and machine learning models to analyze data, improve decision-making, and enhance patient care.
        - text: It encompasses a wide range of applications, from diagnostics and treatment planning to administrative tasks and patient monitoring.
        - text: Example: AI-powered tools like IBM Watson are used for cancer diagnosis and treatment recommendations.
  - key_point:
      key_applications_of_ai_in_healthcare:
        - text: AI is applied in various healthcare sectors, including diagnostics, drug discovery, personalized medicine, and robotic surgery.
       - text: For instance, AI models can analyze medical images to detect anomalies such as tumors or fractures with high accuracy.
        - text: Example: AI systems like DeepMind's AlphaFold have revolutionized protein st

In [15]:
print(result1['detailed_slides'][1])

[{'key_point': 'Data Privacy and Security Concerns', 'explanation': 'AI systems rely heavily on large datasets, often containing sensitive personal information. Mismanagement or breaches can lead to identity theft, fraud, or other privacy violations. For instance, in 2020, a cybersecurity breach at a major healthcare provider exposed the personal data of over 10 million patients.', 'examples': 'Example: In 2020, a cybersecurity breach at a major healthcare provider exposed the personal data of over 10 million patients.'}, {'key_point': 'Ethical and Legal Issues', 'explanation': "AI's decision-making processes can lack transparency, raising ethical concerns about accountability and bias. For example, AI algorithms used in hiring have been shown to favor certain demographics, leading to accusations of discrimination. Additionally, legal frameworks often lag behind technological advancements, creating regulatory uncertainty.", 'examples': 'Example: AI algorithms used in hiring have been s

In [26]:
print(result1['detailed_slides'][2])

{'slide_number': 1, 'slide_title': 'Future of AI in Healthcare', 'detailed_content': [{'key_point': '**Conclusion and Call to Action**', 'content': ['AI has the potential to revolutionize healthcare by improving patient outcomes, reducing costs, and increasing efficiency. As we look to the future, it is essential for stakeholders to collaborate and invest in AI technologies to address current challenges and unlock its full potential.', 'To make this vision a reality, we must prioritize ethical AI development, ensure data security and privacy, and foster interdisciplinary partnerships. By doing so, we can create a healthcare system that is more accessible, equitable, and effective for all.']}]}


In [27]:
print(result1['outline']['slides'][2])

{'slide_number': 3, 'slide_title': 'Future of AI in Healthcare', 'key_points': ['Predictions and Trends', 'Potential Solutions to Current Challenges', 'Conclusion and Call to Action'], 'content_type': 'conclusion'}


In [28]:
ritu = {"ritu":"ritu"}

In [31]:
if ritu.get('rituji'):
    print('hi')